# TensorFlowLinear CPU training

A single linear layer learns y = 2x + 1. Data preparation, model construction, training, evaluation and checkpoint restoration use the public Bovi contracts.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from bovi_core.config import Config
from bovi_core.ml import EvaluationContext, TrainingContext
from bovi_core.ml.models.checkpoints import LocalCheckpointResolver
from tensorflow_linear import (
    TensorFlowLinearEvaluationConfig,
    TensorFlowLinearEvaluator,
    TensorFlowLinearModelConfig,
    TensorFlowLinearModelProvider,
    TensorFlowLinearTrainer,
    TensorFlowLinearTrainingConfig,
    create_dataloader,
)

Config.reset()
config = Config(experiment_name="tensorflow_linear", project_name="tensorflow-linear")
model_config = TensorFlowLinearModelConfig.from_config(config)
training_config = TensorFlowLinearTrainingConfig.from_config(config)
loaders = {
    split: create_dataloader(config, model_config, split) for split in ("train", "validation")
}
print(model_config)
print(next(iter(loaders["train"])))

In [ ]:
output = Path(gettempdir()) / "bovi-tensorflow-linear" / str(uuid4())
provider = TensorFlowLinearModelProvider()
model = provider.create(model_config)
context = TrainingContext(run_id=uuid4(), output_dir=output / "first")
result = TensorFlowLinearTrainer(model, loaders, training_config, context).train()
assert result.status == "completed", result.issues
print(result.status, result.stop_reason)
print("First epoch:", result.epochs[0].metrics)
print("Last epoch:", result.epochs[-1].metrics)
assert result.epochs[-1].metrics["train_mse"] < 0.001
print("Best epoch:", result.best_epoch)

In [ ]:
import json

from bovi_core.ml.trainers import LocalTrainingResultLogger, TrainingContext, TrainingResult

run_metadata = {
    "experiment": "tensorflow_linear",
    "splits": {
        split: {
            "records": loader.num_samples,
            "batches": len(loader),
            "batch_size": loader.batch_size,
        }
        for split, loader in loaders.items()
    },
}
logger = LocalTrainingResultLogger(
    metadata=run_metadata,
    config_snapshot={
        "model": model_config.model_dump(mode="json"),
        "training": training_config.model_dump(mode="json"),
    },
)
log_outcome = await logger.log(context, result)
print(log_outcome.model_dump_json(indent=2))
assert log_outcome.destinations[0].status == "success", log_outcome
manifest_path = context.output_dir / "training-results" / f"{context.run_id}.json"
saved_manifest = json.loads(manifest_path.read_text())
saved_context = TrainingContext.model_validate(saved_manifest["context"])
saved_result = TrainingResult.model_validate(saved_manifest["result"])
assert saved_context == context
assert saved_result == result
assert saved_manifest["config_snapshot"]["training"] == training_config.model_dump(mode="json")
print("Stored manifest:", manifest_path)

In [ ]:
evaluation = TensorFlowLinearEvaluator(
    model, TensorFlowLinearEvaluationConfig.from_config(config)
).evaluate(
    loaders["validation"],
    EvaluationContext(
        evaluation_id=uuid4(),
        split="validation",
        model_version="last",
        training_run_id=context.run_id,
        output_dir=output / "evaluation",
    ),
)
assert evaluation.status == "completed", evaluation.issues
print(evaluation.metrics)
print("Prediction at x=0.5 (expected 2):", model([[0.5]]))

## Resume

Load the last checkpoint into a new model. A new attempt gets its own run ID and starts at epoch 1. Best and last checkpoints stay on disk, while results contain references.

In [ ]:
reference = saved_result.last_checkpoint
restored = provider.restore_checkpoint(
    model_config,
    LocalCheckpointResolver().resolve(reference),
)
resume_context = TrainingContext(
    run_id=uuid4(), resumed_from_run_id=context.run_id, output_dir=output / "resume"
)
resume_config = TensorFlowLinearTrainingConfig(epochs=2)
resumed = TensorFlowLinearTrainer(restored, loaders, resume_config, resume_context).train()
assert resumed.status == "completed", resumed.issues
assert resumed.epochs[0].epoch == 1
print(resumed.epochs[-1].metrics)
Config.reset()

In [ ]:
resume_logger = LocalTrainingResultLogger(
    metadata=run_metadata,
    config_snapshot={
        "model": model_config.model_dump(mode="json"),
        "training": resume_config.model_dump(mode="json"),
    },
)
resume_log_outcome = await resume_logger.log(resume_context, resumed)
print(resume_log_outcome.model_dump_json(indent=2))
assert resume_log_outcome.destinations[0].status == "success", resume_log_outcome
resume_manifest_path = (
    resume_context.output_dir / "training-results" / f"{resume_context.run_id}.json"
)
resume_manifest = json.loads(resume_manifest_path.read_text())
assert resume_manifest["context"]["resumed_from_run_id"] == str(context.run_id)
assert resume_manifest["config_snapshot"]["training"] == resume_config.model_dump(mode="json")
assert TrainingResult.model_validate(resume_manifest["result"]) == resumed
print("Stored resume manifest:", resume_manifest_path)